# データベースの基礎 第3回（参考資料）：制約を体験する
* データベースの意味記述
* 実際のデータベースで動作を確認します
* SQLiteという軽量のDBMSを使用します（Google Colabでそのまま動きます）
* 制約に違反するデータを入れようとすると**エラーになる**ことを確かめます．1〜5のセルはエラーが出るのが正しい動作です

## Google Colab の使い方（初めての人へ）

**開き方**

1. Google アカウントにログインした状態で，学生ポータルの第3回ページにあるリンクを開く（このノートブックが Google Colab で開く）
2. 「警告：このノートブックは Google が作成したものではありません」と出たら「このまま実行」を選ぶ
3. 自分用に保存したいときは，メニュー「ファイル → ドライブにコピーを保存」を選ぶ（保存しなくても実行はできる）

**基本操作**

| 操作 | 方法 |
|------|------|
| セルの実行 | セルをクリックして `Shift + Enter`，またはセル左の再生ボタン |
| すべて実行 | メニュー「ランタイム → すべてのセルを実行」 |
| セルの追加 | 上部メニューの「+ コード」または「+ テキスト」 |

**注意**

* セルは**上から順に**実行する．最初の「環境作成」のセルを飛ばすと，あとのセルがエラーになる
* しばらく操作しないとランタイム（実行環境）が切断される．切断されたら「ランタイム → ランタイムを再起動」のあと，もう一度上から実行する
* このノートブックの「制約を体験する」の 1〜5 のセルは，**エラーが出るのが正しい動作**（制約が働いている証拠）

## 環境作成
* Google Colabで使うときは，最初に以下のセルでjupysqlをインストールしてください
* jupysqlはセルにSQLを入れて実行できる仕組みです
* SQLiteはインストールなしで使えます

In [ ]:
# jupysqlをインストール
%pip install jupysql

### JupySQL拡張をロード

In [ ]:
%load_ext sql

### SQLite（メモリDB）に接続

In [ ]:
# SQLite（メモリDB）に接続
%sql sqlite:///:memory:

## 制約を体験する
### 例題とするテーブルを作成
* 部署テーブル
* 従業員テーブル

In [ ]:
%%sql
PRAGMA foreign_keys = ON;

-- 部署テーブル（外部キーの参照先）
CREATE TABLE department (
    dept_id     INTEGER PRIMARY KEY,
    dept_name   TEXT UNIQUE NOT NULL
);

-- 従業員テーブル（制約まとめ）
CREATE TABLE employee (
    emp_id      INTEGER PRIMARY KEY,                      -- 主キー制約
    name        TEXT NOT NULL,                            -- NOT NULL制約
    email       TEXT UNIQUE,                              -- UNIQUE制約
    salary      REAL CHECK (salary >= 0),                 -- 検査制約
    dept_id     INTEGER REFERENCES department(dept_id),   -- 外部キー制約
    age         INTEGER CHECK (age BETWEEN 18 AND 65)     -- 検査制約
);

### 部署にレコード（タプル）を作成
* 外部キー制約があるので，従業員より先に部署を作成する必要がある

In [ ]:
%%sql
INSERT INTO department VALUES (1, 'Information');
INSERT INTO department VALUES (2, 'Business');

### 従業員にレコード（タプル）を作成

In [ ]:
%%sql
INSERT INTO employee VALUES (1, 'Suzuki', 'suzuki@example.com', 3000, 1, 25);

### できたレコードを確認

In [ ]:
%%sql
SELECT * FROM employee ;

### できたレコードを確認

In [ ]:
%%sql
SELECT * FROM department;

### 1. 主キー制約違反を体験
* 同じ主キーで登録しようとすると失敗する

In [ ]:
%%sql
INSERT INTO employee VALUES (1, 'Duplicate', 'dup@example.com', 2500, 1, 28);

### 2. NOT NULL制約違反を体験
* 名前がNULLだとエラー

In [ ]:
%%sql
INSERT INTO employee VALUES (2, NULL, 'tanaka@example.com', 2800, 2, 30);

### 3. UNIQUE制約違反を体験
* メールアドレスが重複するとエラー

In [ ]:
%%sql
INSERT INTO employee VALUES (3, 'Tanaka', 'suzuki@example.com', 2800, 2, 30);

### 4. CHECK制約違反を体験する
* 給与がマイナスだとエラー

In [ ]:
%%sql
INSERT INTO employee VALUES (4, 'Kato', 'kato@example.com', -100, 1, 22);

### 5. 外部キー制約違反を体験
* 存在しない部署（99）の従業員を作成しようとするとエラー

In [ ]:
%%sql
INSERT INTO employee VALUES (5, 'Sato', 'sato@example.com', 2000, 99, 24);

### トリガを作成する
* 給料変更時（古い給料と新しい給料が等しくない）にログに古い給料と新しい給料を記録する
* ログのテーブルも作成する

In [ ]:
%%sql
CREATE TABLE salary_log (
    emp_id INTEGER,
    old_salary REAL,
    new_salary REAL,
    changed_at TEXT
);

CREATE TRIGGER log_salary_update
AFTER UPDATE OF salary ON employee
FOR EACH ROW
WHEN OLD.salary <> NEW.salary
BEGIN
    INSERT INTO salary_log VALUES (
        OLD.emp_id,
        OLD.salary,
        NEW.salary,
        datetime('now')
    );
END;

### 6. トリガの動作を確認
* 給料を変更するとログに記録される．
* 給料を3000から3500に更新
* ログを確認する

In [ ]:
%%sql
UPDATE employee SET salary = 3500 WHERE emp_id = 1;


In [ ]:
%%sql
SELECT * FROM salary_log;